# Regex in Python: learning through real use cases

A regular expression (regex) is a small language for describing patterns in text. Questions like "where are the 11 digit numbers in this message?" or "which of these words are email addresses?" are easy for a person to answer by looking, but a program needs a pattern to follow. That's what regex gives us.

Instead of going through every symbol one by one, this notebook works through problems you would actually run into: pulling an order number out of a customer message, cleaning up phone numbers, extracting a birth date from a Wikipedia paragraph, cleaning social media text, and normalizing Bangla text for a text-to-speech model.

What's covered:

1. Basics: the `re` module, raw strings, a cheat sheet, groups, greedy vs lazy
2. Use cases
   - Order numbers from a chatbot message
   - Extracting and validating emails
   - Bangladeshi phone numbers
   - Birth date and age from Wikipedia text
   - Cleaning social media text
   - Bangla text and Unicode
   - Splitting sentences
   - A small text normalizer for TTS
3. Common mistakes, practice problems and solutions

Run each cell, then change the pattern or the input and run it again. Breaking a pattern on purpose is the fastest way to understand it.

## Part 1: Basics

### The `re` module

These are the functions you'll use most of the time:

| Function | What it does | Returns |
|---|---|---|
| `re.findall(p, text)` | finds every match | list |
| `re.search(p, text)` | finds the first match anywhere in the text | Match or `None` |
| `re.match(p, text)` | checks only at the start of the text | Match or `None` |
| `re.fullmatch(p, text)` | checks if the whole text fits the pattern | Match or `None` |
| `re.sub(p, new, text)` | replaces matches | new string |
| `re.split(p, text)` | splits the text wherever the pattern matches | list |
| `re.compile(p)` | prepares a pattern once so you can reuse it | Pattern object |

In [1]:
import re

text = "Amar order #2545, dam 1200 taka"

print("findall  :", re.findall(r"\d+", text))

m = re.search(r"\d+", text)
print("search   :", m.group(), "| position", m.start(), "to", m.end())

print("match    :", re.match(r"\d+", text))         # None, the text starts with "Amar"
print("fullmatch:", re.fullmatch(r"\d+", "2545"))    # the whole string is digits
print("sub      :", re.sub(r"\d+", "XXX", text))
print("split    :", re.split(r",\s*", text))

findall  : ['2545', '1200']
search   : 2545 | position 12 to 16
match    : None
fullmatch: <re.Match object; span=(0, 4), match='2545'>
sub      : Amar order #XXX, dam XXX taka
split    : ['Amar order #2545', 'dam 1200 taka']


### Why the `r` in `r"..."`?

When I first ran these examples without it, Python printed warnings like:
```
SyntaxWarning: invalid escape sequence '\d'
```
Backslashes mean something in normal Python strings (`\n` is a newline, `\t` is a tab, `\b` is a backspace). Regex also uses backslashes (`\d`, `\b`, ...). Writing the pattern as a raw string, `r"..."`, tells Python to leave the backslashes alone and pass them straight to the regex engine.

The `\d` case only gives a warning. The `\b` case is worse because it silently gives the wrong answer:

In [2]:
sentence = "cat concat cat"

print(re.findall("\bcat\b", sentence))    # [] because Python turned \b into a backspace character
print(re.findall(r"\bcat\b", sentence))   # ['cat', 'cat'], whole word only, not the one inside "concat"

[]
['cat', 'cat']


So the habit is simple: always write patterns as raw strings.

### Cheat sheet

| Symbol | Meaning | Example | Matches |
|---|---|---|---|
| `.` | any character except newline | `c.t` | cat, cut, c9t |
| `\d` | a digit | `\d\d` | 25, 07 |
| `\D` | not a digit | `\D+` | abc |
| `\w` | letter, digit or `_` | `\w+` | hello_2 |
| `\s` | whitespace | `a\sb` | "a b" |
| `[abc]` | one of a, b, c | `[aeiou]` | a vowel |
| `[a-z]` | a range | `[A-Z]` | a capital letter |
| `[^abc]` | anything except a, b, c | `[^#]` | anything but # |
| `*` | 0 or more | `\d*` | "", 5, 555 |
| `+` | 1 or more | `\d+` | 5, 555 |
| `?` | 0 or 1 (optional) | `colou?r` | color, colour |
| `{n}`, `{n,m}` | exactly n, or n to m times | `\d{11}` | 11 digits |
| `^`, `$` | start / end of line | `^Hi` | line starting with Hi |
| `\b` | word boundary | `\bcat\b` | the word "cat" |
| `a\|b` | a or b | `tk\|taka` | tk, taka |
| `( )` | capture group | `#(\d+)` | just the number |
| `(?: )` | group without capturing | `(?:\+88)?` | optional +88 |
| `\` | escape a special character | `\.` `\+` `\(` | ., +, ( |

In [3]:
demo = "Rahim 25, Karim 30, phone: 01711-223344"

patterns = [r"\d", r"\d+", r"\d{2}", r"\w+", r"[A-Z]\w*", r"[^\w\s]", r"\d+-\d+"]
for p in patterns:
    print(f"{p:12} -> {re.findall(p, demo)}")

\d           -> ['2', '5', '3', '0', '0', '1', '7', '1', '1', '2', '2', '3', '3', '4', '4']
\d+          -> ['25', '30', '01711', '223344']
\d{2}        -> ['25', '30', '01', '71', '22', '33', '44']
\w+          -> ['Rahim', '25', 'Karim', '30', 'phone', '01711', '223344']
[A-Z]\w*     -> ['Rahim', 'Karim']
[^\w\s]      -> [',', ',', ':', '-']
\d+-\d+      -> ['01711-223344']


### Groups

If your pattern has a group in it, `findall` returns only what's inside the group. With more than one group you get tuples.

In [4]:
s = "order #2545 and order #3636"

print(re.findall(r"order #\d+", s))        # whole match
print(re.findall(r"order #(\d+)", s))      # only the number
print(re.findall(r"(order) #(\d+)", s))    # two groups, so tuples

m = re.search(r"order #(\d+)", s)
print(m.group(0), "|", m.group(1))         # group(0) is the full match

['order #2545', 'order #3636']
['2545', '3636']
[('order', '2545'), ('order', '3636')]
order #2545 | 2545


### Greedy vs lazy

`*` and `+` are greedy by default, they take as much text as they can. Add a `?` after them and they become lazy, taking as little as possible.

In [5]:
html = "<b>bold</b> and <i>italic</i>"

print("greedy:", re.findall(r"<.*>", html))    # from the first < all the way to the last >
print("lazy  :", re.findall(r"<.*?>", html))   # each tag separately

greedy: ['<b>bold</b> and <i>italic</i>']
lazy  : ['<b>', '</b>', '<i>', '</i>']


## Part 2: Use cases

### 1. Getting the order number out of a customer message

Say we're building a support chatbot. A customer writes "i am having issues with my order #2425" and the bot needs to pull out the order number so it can look it up.

Here's my first attempt:

In [6]:
query = "i am havin issues with my order number #2545 and order #3636"
pattern = r"order[^d]*#(\d*)"
re.findall(pattern, query)

['2545', '3636']

It works, but only by luck. `[^d]` means "any character except the **letter** d". What I meant was "anything except a digit", which is `[^\d]` or `\D`. It happens to work here because the word "number" has no letter d in it.

There's a second problem: `\d*` can match zero digits, so it will happily return an empty string.

A few messages that expose both problems:

In [7]:
old = r"order[^d]*#(\d*)"

print(re.findall(old, "my order dated yesterday #7788 hasn't arrived"))  # [] because "dated" has a d
print(re.findall(old, "order # still pending"))                          # [''] an empty order number
print(re.findall(old, "Order #9001 kothay?"))                            # [] capital O

[]
['']
[]


The fix:

- `[^#\d]*` means "skip anything until a `#` or a digit", which is what I wanted in the first place
- `\d+` requires at least one digit
- `re.IGNORECASE` handles Order, ORDER, order

In [8]:
pattern = r"order[^#\d]*#(\d+)"

tests = [
    "i am havin issues with my order number #2545 and order #3636",
    "i am having issues with my order #2425",
    "having problem with my order #252. 019 is my phone no",   # 019 shouldn't be picked up
    "my order dated yesterday #7788 hasn't arrived",
    "order # still pending",
    "Order #9001 kothay?",
]
for t in tests:
    print(f"{t[:55]:57} -> {re.findall(pattern, t, flags=re.IGNORECASE)}")

i am havin issues with my order number #2545 and order    -> ['2545', '3636']
i am having issues with my order #2425                    -> ['2425']
having problem with my order #252. 019 is my phone no     -> ['252']
my order dated yesterday #7788 hasn't arrived             -> ['7788']
order # still pending                                     -> []
Order #9001 kothay?                                       -> ['9001']


Two things to remember from this one: `d` and `\d` are very different, and use `+` when you need at least one of something. Also, test with inputs that try to break the pattern, not only the one example you had in mind.

### 2. Extracting and validating email addresses

Two related jobs: find every email in a block of text (a CV, a web page, comments), and check whether the email someone typed into a sign up form looks valid.

In [9]:
query = "abc@gmail.com 132king@gmail.com pan_kha@outlook.com.bd abc"
old = r"[a-z0-9_]*@[a-z.]*"
re.findall(old, query)

['abc@gmail.com', '132king@gmail.com', 'pan_kha@outlook.com.bd']

Fine for this input, but look at what the pattern allows and doesn't allow:

- `[a-z0-9_]` has no capital letters, and no `.` or `+`, which are common in real addresses (`rahim.khan@`, `me+news@`)
- `*` means there can be nothing before or after the `@`, so `@handle` counts as an email
- `[a-z.]*` will grab a trailing full stop at the end of a sentence

In [10]:
tricky = "Contact: Rahim.Khan@Gmail.com, me+news@yahoo.com. Or ping @handle on Twitter"
print(re.findall(old, tricky))

['han@', 'news@yahoo.com.', '@handle']


A better version, read left to right:

```
[A-Za-z0-9._%+-]+   @   [A-Za-z0-9.-]+   \.   [A-Za-z]{2,}
   username              domain          dot   com, bd, org...
```

Note the `\.` before the last part. A plain `.` would mean "any character".

In [11]:
EMAIL = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"

print(re.findall(EMAIL, query))
print(re.findall(EMAIL, tricky))

['abc@gmail.com', '132king@gmail.com', 'pan_kha@outlook.com.bd']
['Rahim.Khan@Gmail.com', 'me+news@yahoo.com']


With two groups we can split the username from the domain, for example to see how many people use Gmail:

In [12]:
users = "abc@gmail.com, 132king@Gmail.com, pan_kha@outlook.com.bd, rimi@du.ac.bd"

pairs = re.findall(r"([A-Za-z0-9._%+-]+)@([A-Za-z0-9.-]+\.[A-Za-z]{2,})", users)
for user, domain in pairs:
    print(f"{user:10} @ {domain}")

gmail = [u for u, d in pairs if d.lower() == "gmail.com"]
print("\nGmail users:", gmail)

abc        @ gmail.com
132king    @ Gmail.com
pan_kha    @ outlook.com.bd
rimi       @ du.ac.bd

Gmail users: ['abc', '132king']


For validating a form field, use `fullmatch`, not `search`. With `search`, an input like `"hello abc@gmail.com bye"` would pass because there's an email somewhere inside it.

Long patterns get hard to read, so I've also used `re.VERBOSE` here. It lets you put spaces and comments inside the pattern.

In [13]:
EMAIL_VERBOSE = re.compile(r"""
    [A-Za-z0-9._%+-]+     # username
    @
    [A-Za-z0-9.-]+        # domain (gmail, outlook.com)
    \.[A-Za-z]{2,}        # ending (.com, .bd)
""", re.VERBOSE)

def is_valid_email(s):
    return EMAIL_VERBOSE.fullmatch(s) is not None

for s in ["rahim@gmail.com", "rahim@gmail", "hello abc@gmail.com bye", "@handle", "a.b+c@du.ac.bd"]:
    print(f"{s:26} -> {is_valid_email(s)}")

rahim@gmail.com            -> True
rahim@gmail                -> False
hello abc@gmail.com bye    -> False
@handle                    -> False
a.b+c@du.ac.bd             -> True


A note on this: a regex that covers every valid email address is huge and not worth writing. In practice you do a basic check like this one and then send a verification link.

### 3. Bangladeshi phone numbers

People write the same number in many ways: `+880...`, `(880)...`, `018-790-...`, `017...`. Before saving them we want them all in one format.

First version, using `|` to list the four formats:

In [14]:
query = """
+8801879093418
(880)1879093418
018-790-93418
01879093418
"""
old = r"\d{11}|\(\d{3}\)\d{10}|\d{3}-\d{3}-\d{5}|\+\d{13}"
re.findall(old, query)

['+8801879093418', '(880)1879093418', '018-790-93418', '01879093418']

`(`, `)` and `+` are special characters in regex, which is why they're written as `\(`, `\)` and `\+`.

Two problems with this pattern though:

1. Nothing stops it from matching inside a longer number. A 15 digit invoice ID gets cut down to 11 digits and treated as a phone number.
2. It doesn't check the operator code. Bangladeshi mobile numbers start with 013 to 019, so `012...` isn't valid.

In [15]:
bad = "Invoice ID 123456789012345, wrong number 01234567890"
re.findall(old, bad)       # both wrong

['12345678901', '01234567890']

#### Lookarounds

A lookaround checks what comes before or after the match without including it in the match.

| Syntax | Name | Meaning |
|---|---|---|
| `(?=X)` | lookahead | followed by X |
| `(?!X)` | negative lookahead | not followed by X |
| `(?<=X)` | lookbehind | preceded by X |
| `(?<!X)` | negative lookbehind | not preceded by X |

So `(?<!\d)\d{11}(?!\d)` means 11 digits with no digit right before or right after.

In [16]:
long_num = "ID 123456789012345 | phone 01711223344"

print(re.findall(r"\d{11}", long_num))                 # cuts a piece out of the long ID
print(re.findall(r"(?<!\d)\d{11}(?!\d)", long_num))    # only the real 11 digit number

['12345678901', '01711223344']
['01711223344']


In my Bangla TTS project I used this pattern for phone numbers:

```
(?<!\d)   (?:\+?88)?   01[3-9]   \d{2}   [-\s]?   \d{6}   (?!\d)
no digit  optional     operator  2       optional  6       no digit
before    +88 or 88    013-019   digits  - or space digits  after
```

In [17]:
BD_PHONE = r"(?<!\d)(?:\+?88)?01[3-9]\d{2}[-\s]?\d{6}(?!\d)"

text = "Call 01711-223344 or +8801812345678. Invoice 123456789012345. Wrong: 01234567890"
re.findall(BD_PHONE, text)

['01711-223344', '+8801812345678']

#### A cleaner approach: clean first, then check

Instead of one giant pattern that handles every format, split the job in two:

1. Find anything that looks roughly like a phone number (digits with `+`, `-`, brackets)
2. Throw away everything that isn't a digit with `re.sub(r"\D", "", ...)`
3. Check the remaining digits with `fullmatch` and return them in a standard format

This is a lot easier to maintain when someone comes along with a new format.

In [18]:
def extract_bd_phones(text):
    result = []
    for cand in re.findall(r"(?<!\d)\+?\(?\d[\d()\-]{8,16}\d(?!\d)", text):
        digits = re.sub(r"\D", "", cand)                       # (880)187... becomes 880187...
        m = re.fullmatch(r"(?:88)?(01[3-9]\d{8})", digits)      # country code is optional
        if m:
            result.append("+88" + m.group(1))
    return result

messy = """
+8801879093418
(880)1879093418
018-790-93418
01879093418
Invoice 123456789012345, wrong number 01234567890
"""
extract_bd_phones(messy)

['+8801879093418', '+8801879093418', '+8801879093418', '+8801879093418']

When you're matching numbers, you almost always want the `(?<!\d) ... (?!\d)` boundaries. And you don't have to do everything in one regex. Mixing a simple regex with a bit of normal Python is often the better design.

### 4. Birth date and age from Wikipedia text

This is a small information extraction task: given the opening paragraph of a biography, find the birth date and work out the person's age.

In [19]:
salman = """
Salman Khan[a] (pronounced [səlˈmaːn xaːn] ⓘ; born 27 December 1965) is an Indian actor, film producer, and television personality who works primarily in Hindi cinema. In a career spanning over three decades, his awards include two National Film Awards as a film producer, and two Filmfare Awards as an actor.
"""

musk = """
Elon Reeve Musk (/ˈiːlɒn/ ⓘ EE-lon; born June 28, 1971) is a businessman, industrialist, and former public official who is the chief executive officer (CEO) and largest shareholder of Tesla and SpaceX.
"""

My first version:

In [20]:
old = r"born ([a-zA-Z0-9,-\/]* [a-zA-Z0-9,-\/]* [a-zA-Z0-9,-\/]*)\)"

for bio in [salman, musk]:
    found = re.findall(old, bio)
    year = re.findall(r"\d{4}", found[0])
    print(found[0], "->", "age =", 2026 - int(year[0]))

27 December 1965 -> age = 61
June 28, 1971 -> age = 55


The output looks right, but there are two mistakes hiding in it.

**A `-` inside square brackets makes a range.** I wrote `[,-\/]` meaning "comma, hyphen or slash". But inside `[ ]`, `,-/` is read as "every character from `,` to `/`", which is `,` `-` `.` `/`. The full stop got in without me asking for it. To get a literal hyphen, put it at the start or end: `[,/-]`.

In [21]:
sample = "a,b-c.d/e+f"
print("[,-/] ->", re.findall(r"[,-/]", sample))   # the . is in there too
print("[,/-] ->", re.findall(r"[,/-]", sample))   # only , / -

[,-/] -> [',', '-', '.', '/']
[,/-] -> [',', '-', '/']


**The age is wrong.** 2026 minus 1965 is 61, but his birthday is on 27 December, which hasn't come yet this year, so he's actually 60. Subtracting years isn't enough.

Here's a better version. The two date formats ("27 December 1965" and "June 28, 1971") get their own patterns, and I've used named groups, `(?P<name>...)`, so the code can say `m["year"]` instead of remembering which number the group was. The age calculation is done with `datetime`.

In [22]:
from datetime import date, datetime

BIRTH_PATTERNS = [
    r"born (?P<day>\d{1,2}) (?P<month>[A-Z][a-z]+),? (?P<year>\d{4})",   # 27 December 1965
    r"born (?P<month>[A-Z][a-z]+) (?P<day>\d{1,2}),? (?P<year>\d{4})",   # June 28, 1971
]

def find_birth_date(text):
    for p in BIRTH_PATTERNS:
        m = re.search(p, text)
        if m:
            return datetime.strptime(f'{m["day"]} {m["month"]} {m["year"]}', "%d %B %Y").date()
    return None

def calc_age(born, today=None):
    today = today or date.today()
    had_birthday = (today.month, today.day) >= (born.month, born.day)
    return today.year - born.year - (0 if had_birthday else 1)

for name, bio in [("Salman Khan", salman), ("Elon Musk", musk)]:
    born = find_birth_date(bio)
    print(f"{name:12} born {born}  ->  age {calc_age(born)}  (today: {date.today()})")

Salman Khan  born 1965-12-27  ->  age 60  (today: 2026-10-04)
Elon Musk    born 1971-06-28  ->  age 55  (today: 2026-10-04)


Use regex to pull the text out, and leave the date arithmetic to a proper library.

### 5. Cleaning social media text

Before running sentiment analysis or feeding messages to a chatbot, you usually need to tidy them up. Some of these rules come from the `clean_text()` function in my TTS project.

#### Backreferences

`([A-Za-z])\1{2,}` means: capture one letter as group 1, then match that same letter 2 or more times again. In the replacement, `\1` puts the captured letter back.

In [23]:
msg = "valooooo laglo!!! sooo goood"

print(re.sub(r"([A-Za-z])\1{2,}", r"\1", msg))      # 3+ repeats become 1
print(re.sub(r"([A-Za-z])\1{2,}", r"\1\1", msg))    # 3+ repeats become 2

valo laglo!!! so god
valoo laglo!!! soo good


The first version turned "goood" into "god", since "good" legitimately has two o's. The second one keeps two letters and is safer. Every cleaning rule has side effects, so check them.

Some other common clean up steps:

In [24]:
post = "Darun offer!!   Visit https://shop.example.com/deal?id=5 ekhoni   #BlackFriday #Sale @rahim_bd   ke tag korlam"

hashtags = re.findall(r"#\w+", post)
mentions = re.findall(r"@\w+", post)
urls     = re.findall(r"https?://\S+", post)       # \S is anything that isn't whitespace

clean = re.sub(r"https?://\S+", "", post)           # remove links
clean = re.sub(r"[#@]\w+", "", clean)               # remove hashtags and mentions
clean = re.sub(r"([!?.])\1+", r"\1", clean)         # "!!" becomes "!"
clean = re.sub(r"\s+", " ", clean).strip()          # collapse extra spaces

print("hashtags:", hashtags)
print("mentions:", mentions)
print("urls    :", urls)
print("clean   :", clean)

hashtags: ['#BlackFriday', '#Sale']
mentions: ['@rahim_bd']
urls    : ['https://shop.example.com/deal?id=5']
clean   : Darun offer! Visit ekhoni ke tag korlam


#### Fixing the Bangla full stop with a lookbehind

A lot of people type `|` because the Bangla dari `।` isn't on their keyboard. We can't replace every `|`, since it could be part of code or a table. So we only replace it when it comes right after a Bangla character:

In [25]:
bn = "আমি ভাত খাই | তুমি কী খাও |  A | B"
print(re.sub(r"(?<=[\u0980-\u09FF])\s*\|", "।", bn))   # "A | B" is left alone

আমি ভাত খাই। তুমি কী খাও।  A | B


### 6. Bangla text and Unicode

Bangla characters live in the Unicode range `\u0980` to `\u09FF`. That covers letters, vowel signs (ি, া, ো) and Bangla digits.

#### `\w` breaks Bangla words apart

This one surprised me. `\w` doesn't count vowel signs like ি or ো as word characters, so `\w+` chops Bangla words into pieces:

In [26]:
s = "আমি কাল বই কিনবো"

print(r"\w+              ->", re.findall(r"\w+", s))
print(r"[\u0980-\u09FF]+ ->", re.findall(r"[\u0980-\u09FF]+", s))

\w+              -> ['আম', 'ক', 'ল', 'বই', 'ক', 'নব']
[\u0980-\u09FF]+ -> ['আমি', 'কাল', 'বই', 'কিনবো']


#### And `\d` matches Bangla digits

In Python 3, `\d` matches any Unicode digit, including ০ to ৯. If you only want English digits, write `[0-9]`.

In [27]:
price = "দাম ৫০০ টাকা, discount 50"

print(r"\d+    ->", re.findall(r"\d+", price))
print(r"[0-9]+ ->", re.findall(r"[0-9]+", price))

# converting Bangla digits to English doesn't need regex, str.translate is simpler and faster
to_ascii = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")
print("converted ->", price.translate(to_ascii))

\d+    -> ['৫০০', '50']
[0-9]+ -> ['50']
converted -> দাম 500 টাকা, discount 50


#### Is a message Bangla, Banglish or mixed?

Useful in a chatbot for deciding how to handle a message:

In [28]:
def detect_script(text):
    has_bn = bool(re.search(r"[\u0980-\u09FF]", text))
    has_en = bool(re.search(r"[A-Za-z]", text))
    if has_bn and has_en:
        return "Mixed"
    if has_bn:
        return "Bangla"
    if has_en:
        return "English/Banglish"
    return "Unknown"

for t in ["আমি ভালো আছি", "ami valo achi", "আমার order টা কই?", "12345"]:
    print(f"{t:20} -> {detect_script(t)}")

আমি ভালো আছি         -> Bangla
ami valo achi        -> English/Banglish
আমার order টা কই?    -> Mixed
12345                -> Unknown


### 7. Splitting text into sentences

Splitting a paragraph into sentences is a common first step before TTS, translation or summarization. I used this when preparing Bangla Wikipedia text.

`(?<=[।?!])\s+` means "split on whitespace that comes right after `।`, `?` or `!`". Because the punctuation is inside a lookbehind, it isn't consumed, so each sentence keeps its ending.

In [29]:
para = "আজ আবহাওয়া ভালো। তুমি কি আসবে? অবশ্যই আসবো! চলো যাই।"

print(re.split(r"[।?!]\s*", para))           # punctuation is lost, plus an empty string at the end
print(re.split(r"(?<=[।?!])\s+", para))      # clean sentences with their punctuation

['আজ আবহাওয়া ভালো', 'তুমি কি আসবে', 'অবশ্যই আসবো', 'চলো যাই', '']
['আজ আবহাওয়া ভালো।', 'তুমি কি আসবে?', 'অবশ্যই আসবো!', 'চলো যাই।']


For English this gets trickier, since splitting on `.` will also break "Dr. Rahman" and "3.5". Libraries like NLTK or spaCy handle those cases better.

### 8. A small text normalizer for TTS

A text-to-speech model can't read "500tk" or "3:30 pm" out loud. They need to become words first: "পাঁচশো টাকা", "বিকাল সাড়ে তিনটা". The `bn_rules.py` file in my TTS project does exactly that. Below is a cut down version of it.

The trick is that `re.sub` accepts a function instead of a replacement string. For each match, it calls the function with the Match object and uses whatever the function returns. That means the replacement can contain real logic.

In [30]:
# simple example: double every number
re.sub(r"\d+", lambda m: str(int(m.group()) * 2), "2 ta apple, 15 ta kola")

'4 ta apple, 30 ta kola'

These helper functions turn numbers into Bangla words. There's no regex in them, just run the cell and move on.

In [31]:
ONES = (
    "শূন্য এক দুই তিন চার পাঁচ ছয় সাত আট নয় "
    "দশ এগারো বারো তেরো চৌদ্দ পনেরো ষোলো সতেরো আঠারো উনিশ "
    "বিশ একুশ বাইশ তেইশ চব্বিশ পঁচিশ ছাব্বিশ সাতাশ আটাশ ঊনত্রিশ "
    "ত্রিশ একত্রিশ বত্রিশ তেত্রিশ চৌত্রিশ পঁয়ত্রিশ ছত্রিশ সাঁইত্রিশ আটত্রিশ ঊনচল্লিশ "
    "চল্লিশ একচল্লিশ বিয়াল্লিশ তেতাল্লিশ চুয়াল্লিশ পঁয়তাল্লিশ ছেচল্লিশ সাতচল্লিশ আটচল্লিশ ঊনপঞ্চাশ "
    "পঞ্চাশ একান্ন বায়ান্ন তিপ্পান্ন চুয়ান্ন পঞ্চান্ন ছাপ্পান্ন সাতান্ন আটান্ন ঊনষাট "
    "ষাট একষট্টি বাষট্টি তেষট্টি চৌষট্টি পঁয়ষট্টি ছেষট্টি সাতষট্টি আটষট্টি ঊনসত্তর "
    "সত্তর একাত্তর বাহাত্তর তিয়াত্তর চুয়াত্তর পঁচাত্তর ছিয়াত্তর সাতাত্তর আটাত্তর ঊনআশি "
    "আশি একাশি বিরাশি তিরাশি চুরাশি পঁচাশি ছিয়াশি সাতাশি অষ্টাশি ঊননব্বই "
    "নব্বই একানব্বই বিরানব্বই তিরানব্বই চুরানব্বই পঁচানব্বই ছিয়ানব্বই সাতানব্বই আটানব্বই নিরানব্বই"
).split()

HOUR_TA = {1: "একটা", 2: "দুইটা", 3: "তিনটা", 4: "চারটা", 5: "পাঁচটা", 6: "ছয়টা",
           7: "সাতটা", 8: "আটটা", 9: "নয়টা", 10: "দশটা", 11: "এগারোটা", 12: "বারোটা"}

def num_to_words(n):
    """1250 -> এক হাজার দুইশো পঞ্চাশ (hajar / lakh / koti style)"""
    if n < 100:
        return ONES[n]
    parts = []
    for size, name in [(10_000_000, " কোটি"), (100_000, " লাখ"), (1000, " হাজার"), (100, "শো")]:
        q, n = divmod(n, size)
        if q:
            parts.append(num_to_words(q) + name)
    if n:
        parts.append(ONES[n])
    return " ".join(parts)

def digits_to_words(s):
    """01711 -> শূন্য এক সাত এক এক (one digit at a time, like a phone number)"""
    return " ".join(ONES[int(ch)] for ch in s if ch.isdigit())

def time_to_words(h, m, ampm=None):
    h24 = h + 12 if (ampm and ampm.lower() == "pm" and h < 12) else h
    period = ("সকাল" if 4 <= h24 < 12 else "দুপুর" if h24 < 15 else
              "বিকাল" if h24 < 18 else "সন্ধ্যা" if h24 < 20 else "রাত")
    h12 = h24 % 12 or 12
    if m == 0:    core = HOUR_TA[h12]
    elif m == 30: core = "সাড়ে " + HOUR_TA[h12]
    elif m == 15: core = "সোয়া " + HOUR_TA[h12]
    elif m == 45: core = "পৌনে " + HOUR_TA[h12 % 12 + 1]
    else:         core = HOUR_TA[h12] + " বেজে " + ONES[m] + " মিনিট"
    return (period + " " + core) if ampm else core

print(num_to_words(1250), "|", digits_to_words("01711"), "|", time_to_words(3, 30, "pm"))

এক হাজার দুইশো পঞ্চাশ | শূন্য এক সাত এক এক | বিকাল সাড়ে তিনটা


Now the rules. Each rule is a compiled pattern paired with a function. Compiling the patterns once with `re.compile` saves work when the normalizer runs over thousands of lines.

The order of the rules matters a lot. If the phone rule ran after the general number rule, `01711223344` would be read out as one huge number. So specific rules go first and the catch-all number rule goes last.

In [32]:
NUM = r"\d[\d,]*"                  # 1250 or 1,250
RULES = []

def rule(pattern):
    """Decorator that adds the function and its compiled pattern to RULES."""
    def deco(fn):
        RULES.append((re.compile(pattern, re.IGNORECASE), fn))
        return fn
    return deco

@rule(r"(?<!\d)(?:\+?88)?01[3-9]\d{2}-?\d{6}(?!\d)")              # phone, must come first
def _phone(m):
    return digits_to_words(m.group())

@rule(r"(?<!\d)(\d{1,2}):(\d{2})\s*(am|pm)?(?![a-z])")            # time, 3:30 pm
def _time(m):
    return time_to_words(int(m[1]), int(m[2]), m[3])

@rule(r"(?:৳|\bTk\.?|\bBDT)\s*(" + NUM + r")")                    # taka with symbol before, ৳500
def _taka_prefix(m):
    return num_to_words(int(m[1].replace(",", ""))) + " টাকা"

@rule(r"(" + NUM + r")\s*(?:tk\b|taka\b|টাকা|৳|/-)")              # taka with symbol after, 500tk
def _taka_suffix(m):
    return num_to_words(int(m[1].replace(",", ""))) + " টাকা"

@rule(r"(" + NUM + r")\s*%")                                      # percent
def _percent(m):
    return num_to_words(int(m[1].replace(",", ""))) + " শতাংশ"

@rule(r"(?<!\d)(\d+)\s*(ta|ti|টা|টি)(?![a-z])")                    # 2ta -> দুইটা
def _classifier(m):
    suffix = {"ta": "টা", "ti": "টি"}.get(m[2].lower(), m[2])
    return num_to_words(int(m[1])) + suffix

@rule(r"(?<![\d,])(\d{1,3}(?:,\d{3})+|\d+)(?![\d,])")             # any other number, must come last
def _integer(m):
    return num_to_words(int(m[1].replace(",", "")))

def normalize(text):
    text = text.translate(str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789"))
    for pattern, fn in RULES:
        text = pattern.sub(lambda m, fn=fn: " " + fn(m) + " ", text)
    text = re.sub(r"\s+([,।?!.])", r"\1", text)      # no space before punctuation
    return re.sub(r"\s+", " ", text).strip()

samples = [
    "Kal 3:30 pm e 500tk diye 2ta boi kinbo",
    "Amar number 01711-223344, call koro 10:15 am e",
    "Discount 25%! Dam ৳1,250 matro",
    "আমার বয়স ২২ বছর, বেতন ৳৪৫০০০",
]
for s in samples:
    print("IN :", s)
    print("OUT:", normalize(s), "\n")

IN : Kal 3:30 pm e 500tk diye 2ta boi kinbo
OUT: Kal বিকাল সাড়ে তিনটা e পাঁচশো টাকা diye দুইটা boi kinbo 

IN : Amar number 01711-223344, call koro 10:15 am e
OUT: Amar number শূন্য এক সাত এক এক দুই দুই তিন তিন চার চার, call koro সকাল সোয়া দশটা e 

IN : Discount 25%! Dam ৳1,250 matro
OUT: Discount পঁচিশ শতাংশ! Dam এক হাজার দুইশো পঞ্চাশ টাকা matro 

IN : আমার বয়স ২২ বছর, বেতন ৳৪৫০০০
OUT: আমার বয়স বাইশ বছর, বেতন পঁয়তাল্লিশ হাজার টাকা 



To see why the order matters, move the phone rule to the end and try again:

In [33]:
phone_rule = RULES.pop(0)     # take the phone rule off the front
RULES.append(phone_rule)      # and put it at the end

print(normalize("Amar number 01711223344"))    # read as one big number

RULES.insert(0, RULES.pop())  # put it back
print(normalize("Amar number 01711223344"))

Amar number একশো একাত্তর কোটি বারো লাখ তেইশ হাজার তিনশো চুয়াল্লিশ
Amar number শূন্য এক সাত এক এক দুই দুই তিন তিন চার চার


This is how regex tends to get used in real NLP work: lots of small rules, applied in a careful order. The full `bn_rules.py` also has rules for dates (`12/03/2025`), ordinals (`1st`), units (`5kg`) and ranges (`10-15`), and they all follow the same structure as above.

## Part 3: Common mistakes, practice and summary

### Mistakes I see (and have made) most often

| Mistake | Example | Better |
|---|---|---|
| Forgetting the raw string | `"\bcat\b"` | `r"\bcat\b"` |
| Mixing up `d` and `\d` | `[^d]` (not the letter d) | `[^\d]` or `\D` |
| Using `*` when you need at least one | `#(\d*)` matches `''` | `#(\d+)` |
| A `-` in the middle of `[ ]` | `[,-/]` also matches `.` | `[,/-]` |
| Not escaping `.` | `gmail.com` also matches `gmailXcom` | `gmail\.com` |
| No boundary around numbers | `\d{11}` cuts into long numbers | `(?<!\d)\d{11}(?!\d)` |
| Greedy `.*` | `<.*>` | `<.*?>` |
| `\w` on Bangla text | `\w+` splits words | `[\u0980-\u09FF]+` |
| Forgetting `\d` matches Bangla digits | `\d+` | `[0-9]+` for English only |
| `search` for validation | `"hi a@b.com"` passes | `fullmatch` |
| Rules in the wrong order | general rule first | specific first, general last |

### Debugging tips

- [regex101.com](https://regex101.com) explains each part of a pattern as you type it. Set the flavor to Python.
- For long patterns, use `re.VERBOSE` and write comments.
- Test edge cases: empty input, capital letters, long numbers, Bangla text.

### Practice

Try these yourself before scrolling down to the solutions.

1. From `"#Dhaka te #rain, #Dhaka traffic jam #rain #rain"`, count how many times each hashtag appears.
2. From `"BDT 1,200 | 850 taka | ৳99 | 45tk"`, get every price as an integer: `[1200, 850, 99, 45]`.
3. From `"Exam 04/10/2026 ar result 15-11-2026"`, get each date as `(day, month, year)`. Both `/` and `-` should work.
4. Write a password check: at least 8 characters, at least one capital letter and at least one digit. Use `fullmatch` and lookaheads `(?=...)`.
5. From `"https://www.prothomalo.com/bangladesh"` and `"http://bdnews24.com"`, get just `prothomalo.com` and `bdnews24.com`.

In [34]:
# your solutions here

### Solutions

In [35]:
from collections import Counter

# 1. hashtag counts
print("1:", Counter(re.findall(r"#\w+", "#Dhaka te #rain, #Dhaka traffic jam #rain #rain")))

# 2. prices as integers
prices = "BDT 1,200 | 850 taka | ৳99 | 45tk"
amounts = re.findall(r"(?:BDT|৳)\s*(\d[\d,]*)|(\d[\d,]*)\s*(?:taka|tk)", prices)
print("2:", [int((a or b).replace(",", "")) for a, b in amounts])

# 3. dates; \2 makes sure both separators are the same
print("3:", re.findall(r"(\d{1,2})([/-])(\d{1,2})\2(\d{4})", "Exam 04/10/2026 ar result 15-11-2026"))

# 4. password; each lookahead checks "somewhere in the string there is..."
STRONG = r"(?=.*[A-Z])(?=.*\d).{8,}"
for pw in ["hello123", "Hello123", "Hi1", "HELLOworld"]:
    print("4:", f"{pw:11}", bool(re.fullmatch(STRONG, pw)))

# 5. domain
for url in ["https://www.prothomalo.com/bangladesh", "http://bdnews24.com"]:
    print("5:", re.search(r"https?://(?:www\.)?([^/]+)", url).group(1))

1: Counter({'#rain': 3, '#Dhaka': 2})
2: [1200, 850, 99, 45]
3: [('04', '/', '10', '2026'), ('15', '-', '11', '2026')]
4: hello123    False
4: Hello123    True
4: Hi1         False
4: HELLOworld  False
5: prothomalo.com
5: bdnews24.com


In solution 3, `findall` returns four items per date because the separator is also a group. If you want only `(day, month, year)`, you can drop it from each tuple, or try rewriting it with `re.finditer` and named groups.

### Summary

- Write patterns as raw strings, `r"..."`.
- `findall` to get everything, `search` to check if something is there, `fullmatch` to validate, `sub` to replace or clean, `split` to break text apart.
- Groups pull out the part you need. Named groups make the code easier to read.
- Put `(?<!\d) ... (?!\d)` around number patterns.
- For Bangla, use `[\u0980-\u09FF]` rather than `\w`.
- Bigger jobs work better as several small rules plus some plain Python than as one huge pattern.
- Use regex to find the text, and normal Python for calculations like dates, ages and prices.